# Préparation des données SFT — MediQAl (mcqu)

Partie française principale du SFT : QCM à **réponse unique** rattachés à un **cas clinique** (configuration `mcqu` de MediQAl, licence CC-BY-4.0).

- **Prompt** : cas clinique, question, puis les propositions A à E. **Réponse** : lettre et texte de la bonne proposition (« C. Pneumonie lobaire aiguë »). L'exactitude des réponses générées se mesure ainsi sur la lettre.
- **Répartition par cas clinique** : un même cas sert à plusieurs questions, et les splits officiels de MediQAl partagent des cas. Chaque cas est affecté à un seul usage (SFT ou DPO) et à un seul split (`projet_14.mediqal`) : aucun cas n'est vu à la fois à l'entraînement et à l'évaluation, ni à la fois en SFT et en DPO.
- **Sortie** : `data/anonymized/sft_mediqal_mcqu_{train,validation,test}.anonymized.jsonl`, au format `messages` du SFT.

# Import

In [1]:
import pandas as pd
import json

from datasets import load_dataset
from datasets import Dataset, DatasetDict

from projet_14.mediqal import (
    OPTION_COLUMNS, OPTIONS_HEADER, SYSTEM_PROMPT, assign_cases, clean_text, correct_letters, format_answer, user_content,
)
from projet_14.rgpd import run

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqu")
df_train = ds["train"].to_pandas()
df_validation = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df_train.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
1,10495,NaN,"Devant une tuméfaction cervicale médiane, juxt...",Kyste dermoïde,Laryngocèle,Kyste du tractus thyréoglosse,Grenouillette sublinguale,Lithiase salivaire,C,QCU,Otorhinolaryngology (ENT),Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning


In [4]:
# Nombres de lignes dans les différentes parties du dataset
print(f"Train dataset: {len(df_train)} rows")
print(f"Validation dataset: {len(df_validation)} rows")
print(f"Test dataset: {len(df_test)} rows")

Train dataset: 10113 rows
Validation dataset: 2561 rows
Test dataset: 4343 rows


In [5]:
# Print distinct values of "medical_subject" column
distinct_medical_subjects = df_train["medical_subject"].unique()
print(f"Distinct values of 'medical_subject' column: {distinct_medical_subjects}")

Distinct values of 'medical_subject' column: <ArrowStringArray>
[             'Infectious Diseases',        'Otorhinolaryngology (ENT)',
     'Endocrinology and Metabolism',                       'Pediatrics',
                     'Microbiology',                      'Orthopedics',
                     'Biochemistry',                       'Cardiology',
        'Gynecology and Obstetrics',                        'Radiology',
                      'Dermatology',                      'Pulmonology',
                       'Psychiatry',                     'Rheumatology',
          'Hepato-Gastroenterology',                       'Hematology',
                        'Neurology',                       'Immunology',
                    'Ophthalmology',                     'Parasitology',
                   'Nephro-Urology',             'Pathological Anatomy',
            'Occupational Medicine', 'Forensic Medicine and Toxicology',
                   'Rehabilitation',                       '

In [6]:
# Les trois splits officiels réunis : le découpage est refait par cas clinique (plus bas)
df = pd.concat(
    [d.assign(official_split=name) for name, d in {"train": df_train, "validation": df_validation, "test": df_test}.items()],
    ignore_index=True,
)
df = df.assign(**{col: df[col].map(clean_text, na_action="ignore") for col in ["clinical_case", "question"] + OPTION_COLUMNS})

# Uniquement les questions rattachées à un cas clinique (contexte patient, proche du triage)
n_before = len(df)
df = df.dropna(subset=["clinical_case", "question"]).drop_duplicates(subset=["clinical_case", "question"])
print(f"{n_before} rows, {len(df)} avec un cas clinique (après dédoublonnage)")

17017 rows, 6919 avec un cas clinique (après dédoublonnage)


# Répartition par cas clinique

MediQAl n'a pas d'identifiant de cas : `assign_cases` en calcule un (hash du texte du cas), puis en tire un **usage** (`sft` ou `dpo`, moitié-moitié) et un **split** (70 % train, 10 % validation, 20 % test). L'affectation ne dépend que du texte du cas : elle est la même dans les trois notebooks MediQAl, pour `mcqu` comme pour `mcqm`.

In [7]:
df = assign_cases(df)

# Cas partagés entre splits officiels : la raison de cette nouvelle répartition
cases_by_split = df.groupby("official_split")["case_id"].agg(set)
print(f"Cas présents à la fois dans le train et le test officiels : {len(cases_by_split['train'] & cases_by_split['test'])}")

pd.crosstab(df["usage"], df["split"], margins=True)

Cas présents à la fois dans le train et le test officiels : 954


split,test,train,validation,All
usage,,,,
dpo,719,2263,392,3374
sft,749,2454,342,3545
All,1468,4717,734,6919


In [8]:
# Ce notebook ne garde que les cas réservés au SFT
splits = {name: df[(df["usage"] == "sft") & (df["split"] == name)] for name in ["train", "validation", "test"]}
for name, d in splits.items():
    print(f"{name}: {len(d)} questions, {d['case_id'].nunique()} cas")

train: 2454 questions, 684 cas
validation: 342 questions, 93 cas
test: 749 questions, 205 cas


In [9]:
splits["train"].head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type,official_split,case_id,usage,split
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning,train,f8d04e810e1e,sft,train
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding,train,e05088544a34,sft,train
7,23271,"Mme B.A âgée de 52 ans, fait une chute sur la ...",(cochez la réponse juste) Cette fracture conso...,25 à 35 jours,35 à 45 jours,45 à 60 jours,60 a 90 jours,Aucune réponse n'est juste,C,QCU,Orthopedics,Understanding,train,d0d7dc311ac8,sft,train
16,1929,"Un homme de 40 ans, représentant de commerce, ...","Parmi les antibiotiques suivants, lequel n'a a...",Pénicilline G,Tétracyclines,Spectinomycine,Erythromycine,Benzathine-pénicilline,C,QCU,Dermatology,Reasoning,train,872493a427cf,sft,train
25,2994,Madame S amène son fils de 18 mois pour une tu...,Compte-tenu de vos réponses aux questions 1 et...,0 %,25 %,50 %,75 %,100 %,C,QCU,Hematology,Reasoning,train,8396ef45b876,sft,train


# Build SFT JSONL file from dataset

In [10]:
def to_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content(row)},  # cas clinique, question, propositions
        {"role": "assistant", "content": format_answer(row, correct_letters(row), as_list=False)},  # « C. ... »
    ]


METADATA = ["id", "source", "case_id", "medical_subject", "question_type", "golden_answer", "lang"]

# Un fichier JSONL par split : ../data/sft_mediqal_mcqu_{train,validation,test}.jsonl
sft_ds = DatasetDict()
for name, df_split in splits.items():
    df_split = df_split.assign(
        source="MediQAl-mcqu", lang="fr", golden_answer=df_split["correct_answers"], messages=df_split.apply(to_messages, axis=1)
    )
    sft_ds[name] = Dataset.from_pandas(df_split[METADATA + ["messages"]], preserve_index=False)
    sft_ds[name].to_json(f"../data/sft_mediqal_mcqu_{name}.jsonl", force_ascii=False)

print(sft_ds["train"][0]["messages"][1]["content"][-400:], end="\n\n")
print(sft_ds["train"][0]["messages"][2]["content"])

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 41.44ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 105.87ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 50.53ba/s]

0,40 (N = 0,27 - 0,61) ; T3 = 350 mg/100 ml (N = 70 - 165). Scintigraphie thyroïdienne : fixation homogène diffuse sur un corps thyroïde globalement augmenté de volume

Le traitement de cette patiente fait appel à :

Propositions :
A. Chirurgie en urgence
B. Iode radio-actif en urgence
C. Iode radio-actif en préparation à la chirurgie
D. Administration d'antithyroïdiens
E. Aucun de ces traitements

D. Administration d'antithyroïdiens


In [11]:
print(sft_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 2454
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 342
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 749
    })
})


# Anonymisation des données

In [12]:
# Anonymisation de chaque split : ../data/anonymized/sft_mediqal_mcqu_{split}.anonymized.jsonl
# Seuls le cas clinique et la question sont anonymisés (message user, jusqu'à l'en-tête des propositions) :
# les propositions et la réponse sont des énoncés médicaux sans donnée patient, où Presidio prend des termes
# médicaux pour des noms (« Héparine IV » -> <PERSON>)
reports = {
    name: run(
        f"../data/sft_mediqal_mcqu_{name}.jsonl", lang="fr", output_dir="../data/anonymized",
        roles=["user"], keep_after=OPTIONS_HEADER,
    )
    for name in sft_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,2454,788,1435
validation,342,129,218
test,749,292,504


In [13]:
# DataFrame des champs "content" des jsonl anonymisés : une ligne par message (hors system)
rows = []
for name in sft_ds:
    with open(f"../data/anonymized/sft_mediqal_mcqu_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for message in record["messages"]:
                if message["role"] != "system":
                    rows.append({"split": name, "id": record["id"], "role": message["role"], "content": message["content"]})

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
contents_df.head()

7090 messages, dont 1209 avec au moins une balise <PERSON>


,split,id,role,content,n_person
0,train,17342,user,Une jeune femme de 24 ans vous consulte pour a...,0
1,train,17342,assistant,D. Administration d'antithyroïdiens,0
2,train,22883,user,"Un nouveau-né de sexe masculin, âgé de 3 jours...",0
3,train,22883,assistant,D. Ictère par incompatibilité ABO,0
4,train,23271,user,"Mme <PERSON> âgée de 52 ans, fait une chute su...",1


In [14]:
import re


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["split", "id", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(20, random_state=42)

1775 occurrences de <PERSON>


,split,id,role,context
280,train,22973,user,s précisent que le nourrisson est asthénique et anorexique. <PERSON> n'a pas d'antécédent pathologique particulier. Ses vaccina
4544,train,2544,user,"consanguinité, mais l'interrogatoire révèle que le frère de <PERSON> est hémophile.<PERSON> est actuellement enceinte, à 4 semai"
4318,train,23313,user,"<PERSON> âgée de 70 ans, diabétique et hypertendue, chute de sa haut"
1340,train,24304,user,2. <PERSON> + cholédocotomie + drainage biliaire externe 3. <PERSON> + cholédocotomie sous coelio-chirurgie + drainage 4. Sphinc
2248,train,5076,user,"Mr <PERSON> 25 ans, est employé aux abattoirs depuis deux mois. Il vien"
4408,train,16804,user,"Monsieur <PERSON> , 80 ans veuf, vivant seul est traité depuis un an pour une"
3264,train,13506,user,"Madame <PERSON> 72 ans, est hospitalisée pour tentative de suicide médicame"
1250,train,15611,user,"Mr <PERSON>, 28 ans, se décide à consulter car il se sent de plus en pl"
1832,train,22966,user,influenzae : 1. Cefotaxime 2. Amoxicilline 3. Oxacilline 4. <PERSON> 5. Gentamycine Propositions : A. 1+2+4 B. 1+2+5 C. 1+3+4 D
1456,train,25817,user,"Dix jours après un accouchement dystocique, madame <PERSON> âgée de 40 ans, <PERSON>, est hospitalisée pour hémorragies"


# Contrôle final

In [15]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement
DATA_FILES = {name: f"../data/anonymized/sft_mediqal_mcqu_{name}.anonymized.jsonl" for name in sft_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

# Aucun cas clinique partagé entre train, validation et test
cases = {name: set(split["case_id"]) for name, split in final_ds.items()}
assert not cases["train"] & (cases["validation"] | cases["test"]), "cas partagé entre train et évaluation"
assert not cases["validation"] & cases["test"], "cas partagé entre validation et test"

# Propositions et réponses intactes après anonymisation
for split in final_ds.values():
    for messages in split["messages"]:
        assert "<PERSON>" not in messages[1]["content"].split(OPTIONS_HEADER)[1] + messages[2]["content"]
print("OK : aucun cas partagé entre les splits, propositions et réponses intactes")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 2454 examples [00:00, 125035.80 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 342 examples [00:00, 75925.05 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 749 examples [00:00, 90209.15 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 2454
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 342
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 749
    })
})
OK : aucun cas partagé entre les splits, propositions et réponses intactes
